In [1]:
import h5py
import finesse
import networkx as nx
import torch
from GNN.power_predictor import LinGNN as PowerGNN
from train_power_predictor import PowerDataset
import torch_geometric as pyg
import numpy as np
from utils.finesse_base import base_kat
from GNN.GNN_utils import model_to_nx_port
from GNN.GNN_run import run_GNN

/home/xuesi.ma/.conda/envs/ligoopt/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# kat = """
# # Add a Laser named L0 with a power of 1 W.
# l L0 P=1

# s s1 portA=L0.p1 portB=eom1.p1 L=10

# modulator eom1 9M 0.1 order=1

# s s2 portA=eom1.p2 portB=ITM.p1 L=10

# # Input mirror of cavity.
# m ITM L=0 T=0.014 Rc=-17

# # Intra-cavity space with length of 4 km
# s CAV ITM.p2 ETM.p1 L=10

# # End mirror of cavity.
# m ETM L=0 T=5u  Rc=21

# cavity cavArm source=ITM.p2.o

# # Power detectors on reflection, circulation and transmission.
# pd ETM_p1_i_pd ETM.p1.i

# # Power detectors on reflection, circulation and transmission.
# pd ITM_p2_i_pd ITM.p2.i

# pd1 pdhI node=ITM.p1.o f=eom1.f phase=0 # In phase demodulated signal
# pd1 pdhQ node=ITM.p1.o f=eom1.f phase=90 # Quadrature phase demodulated signal

# # dof ETMz ETM.dofs.z
# # readout_rf pdh_readout ITM.p1.o f=eom1.f output_detectors=true phase=0

# # Add a lock
# lock lock_length pdhI ETM.phi -1.0673950644453318 1e-12
# """

In [ ]:
# test = """
#         # Add a Laser named L0 with a power of 1 W.
#         l l0 P=1
#         bp roc_l0 l0.p1.o rc

#         # Space attaching L0 <-> m1 with length of 0 m (default).
#         s s0 l0.p1 m1.p1 20

#         # Input mirror of cavity.
#         m m1 R=0.9 T=0.1 Rc=-1934

#         # Intra-cavity space with length of 10 m.
#         s LX m1.p2 m2.p1 L=3994.47

#         # End mirror of cavity.
#         m m2 R=0.9 T=0.1 Rc=2245

#         cav cavity1 m1.p2.o

#         noxaxis()
#         """

In [ ]:
# def reset_model(kat):
#     fabry_perot = finesse.Model()
#     fabry_perot.parse(kat)
#     fabry_perot.modes(maxtem=6, modes='even')
#     return fabry_perot

In [ ]:
# finesse_model = reset_model(kat)
# # finesse_model = reset_model(test)
# # finesse.tb()
# # graph = model_to_nx_port_sanitized(finesse_model)

# # out1 = finesse_model.run("run_locks(display_progress=true,pre_step=print_model_attr(ETM.phi))")
# out = finesse_model.run("noxaxis()")
# print('\n my gain', out['circ'])
graph = model_to_nx_port(base_kat)

In [ ]:
model = PowerGNN(hidden_size=1000, num_layers=10, lin_layers=5, target_size = 1)
model.load_state_dict(torch.load('GNN/power_predictor_ligo_fixed_gat10_kan5.pt', map_location=torch.device('cpu'), weights_only=True))
model.eval()

In [ ]:
data = pyg.utils.from_networkx(
    graph,
    group_node_attrs=['Rc', 'R', 'alpha'],
    group_edge_attrs=['length', 'nr']
)
data.x = torch.nan_to_num(data.x, posinf=0).float()
data.edge_attr = torch.nan_to_num(data.edge_attr, posinf=0).float()
with torch.no_grad():
    out = model(data)

name = [n for n, attrs in graph.nodes(data=True)]

print(out.shape)
for i, node in enumerate(name):
    print(f"Predicted power at node {node}: {np.exp(out[i].item())}")


In [ ]:
# model = reset_model(kat)
# out = model.run("""Series(
#                             run_locks(display_progress=false),
#                             noxaxis(),
#                             )""")
# print("power at ETM_p1_i_pd", out['noxaxis']['ETM_p1_i_pd'])
# print("power at ITM_p2_i_pd", out['noxaxis']['ITM_p2_i_pd'])

In [ ]:
dataset_fp = PowerDataset(data_files=['GNN/models/training_dataset_ligoParams.h5'])

In [ ]:
# dataset_fp = PowerDataset(data_files=['/home/ma/gnn-ifosim-sid/data/fabry_perot_data_fixed.h5'])

In [ ]:
data = dataset_fp.get(0).y
print(data)
print(torch.expm1(data))
# for i in range(10):
#     print(torch.expm1(dataset_fp[i]['pd']))

In [2]:
model_path = "GNN/models/power_predictor_ligoParams_loss_fixed_gat10_kan5.pt"
pd_name = "ETM.p1.i"
names, powers = run_GNN(base_kat,model_path) 
index = names.index(pd_name)
print(powers[index])

277.9486389160156
